In [2]:
import os, sys

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))
from nbafuns import *
from nba_api.stats.endpoints import leaguedashlineups
# import cv2
import matplotlib.image as image
from IPython.core.display import HTML

fig_DIR = "../figs/teams/"
box_DIR = "../data/box/"
img_DIR_T = "../data/images/teams/"
img_DIR_P = "../data/images/players/"

In [3]:
stats = leaguedashteamstats.LeagueDashTeamStats(
    measure_type_detailed_defense="Advanced",
    season_type_all_star="Playoffs",
)
df1 = stats.get_data_frames()[0]
df1 = df1.sort_values("W_PCT",ascending=False).reset_index(drop=True)
df1["OPTS"] = df1["OFF_RATING"]*df1["POSS"]
df1["DPTS"] = df1["DEF_RATING"]*df1["POSS"]
lavg_Rtg = round(df1["OPTS"].sum()/df1["POSS"].sum(),2)

In [4]:
stats = leaguedashlineups.LeagueDashLineups(
    group_quantity=5,
    measure_type_detailed_defense="Advanced",
    per_mode_detailed="Totals",
    pace_adjust="Y",
    plus_minus="Y",
    rank="Y",
    season_type_all_star="Playoffs"
)
df = stats.get_data_frames()[0]

In [26]:
df1 = df.sort_values("MIN",ascending=False).reset_index(drop=True)
df1["OFF_RATING"] = (df1["POSS"]*df1["OFF_RATING"]+ lavg_Rtg*550)/(df1["POSS"]+550)
df1["DEF_RATING"] = (df1["POSS"]*df1["DEF_RATING"]+ lavg_Rtg*850)/(df1["POSS"]+850)
df1["NET_RATING"] = df1["OFF_RATING"] - df1["DEF_RATING"]
df1["OFF_RATING"] = df1["OFF_RATING"].round(1)
df1["DEF_RATING"] = df1["DEF_RATING"].round(1)
df1["NET_RATING"] = df1["NET_RATING"].round(1)
# df2 = df1.nlargest(100,"MIN")
df2 = df1.query("MIN >= 10")
# df2 = df1.copy()
# team = 'LAC'
# df2 = df1.query(f"TEAM_ABBREVIATION == '{team}' & MIN >= 10")
min_min = df2["MIN"].min()
max_min = df2["MIN"].max()
len(df2)

88

In [27]:
var = "NET_RATING"
# var = "OFF_RATING"
# var = "DEF_RATING"
# var = "MIN"
df3 = df2.sort_values(var,ascending=False).reset_index(drop=True)
# df3 = df2.sort_values(var,ascending=True).reset_index(drop=True)
df3.index +=1
df3[["B1","P11","P21","P31","P41","P51","B2"]] = df3["GROUP_ID"].str.split("-",expand=True)
df3["P1"]=img_DIR_P + df3["P11"] + ".png"
df3["P2"]=img_DIR_P + df3["P21"] + ".png"
df3["P3"]=img_DIR_P + df3["P31"] + ".png"
df3["P4"]=img_DIR_P + df3["P41"] + ".png"
df3["P5"]=img_DIR_P + df3["P51"] + ".png"
df3["NET_PER"] = df3["NET_RATING"].rank(pct=True)*100
df3["OFF_PER"] = df3["OFF_RATING"].rank(pct=True)*100
df3["DEF_PER"] = df3["DEF_RATING"].rank(pct=True,ascending=False)*100

In [28]:
df4 = df3[["GROUP_NAME","P1","P2","P3","P4","P5",var,"MIN","NET_PER","OFF_PER","DEF_PER"]].head(10)
# df4 = df3[["GROUP_NAME","P1","P2","P3","P4","P5","MIN","NET_RATING","NET_PER","OFF_PER","DEF_PER"]].head(20)
df4["P1"] = df4["P1"].apply(path_to_image_html)
df4["P2"] = df4["P2"].apply(path_to_image_html)
df4["P3"] = df4["P3"].apply(path_to_image_html)
df4["P4"] = df4["P4"].apply(path_to_image_html)
df4["P5"] = df4["P5"].apply(path_to_image_html)
# df4 = df4.rename(columns={"GROUP_NAME":f"Best {var}"})
df4 = df4.style.set_caption(f'Best {var}').set_table_styles([{
    'selector': 'caption',
    'props': [
        ('color', 'red'),
        ('font-size', '25px'),
        ('font-style', 'italic'),
        ('font-weight', 'bold'),
        ('text-align', 'center')
    ]
}])
df4 = df4.background_gradient(axis=0,subset=["NET_PER","OFF_PER","DEF_PER"], vmin=0, vmax=100, cmap="RdYlGn")
df4 = df4.background_gradient(axis=0,subset=["MIN"], vmin=min_min, vmax=max_min, cmap="RdYlGn")
# df4 = df4.background_gradient(axis=0,subset=["NET_RATING"], vmin=df2["NET_RATING"].min(), vmax=df2["NET_RATING"].max(), cmap="RdYlGn")
df4 = df4.format(formatter={'MIN':'{:,.0f}',var:'{:,.1f}',"NET_PER":'{:,.1f}',"OFF_PER":'{:,.1f}',"DEF_PER":'{:,.1f}'})
# df4 = df4.format(formatter={'MIN':'{:,.0f}',"NET_PER":'{:,.1f}',"OFF_PER":'{:,.1f}',"DEF_PER":'{:,.1f}',"NET_RATING":'{:,.1f}'})
HTML(df4.to_html(escape=False))

,GROUP_NAME,P1,P2,P3,P4,P5,NET_RATING,MIN,NET_PER,OFF_PER,DEF_PER
1,M. Turner - P. Siakam - A. Nembhard - T. Haliburton - A. Nesmith,,,,,,4.9,98,100.0,96.6,98.9
2,R. Westbrook - A. Gordon - N. Jokić - J. Murray - C. Braun,,,,,,4.8,75,98.9,98.9,81.2
3,A. Gordon - N. Jokić - J. Murray - M. Porter Jr. - C. Braun,,,,,,4.4,156,97.7,100.0,59.7
4,A. Caruso - I. Hartenstein - A. Wiggins - J. Williams - C. Wallace,,,,,,3.9,22,96.6,97.7,77.8
5,J. Randle - D. DiVincenzo - N. Reid - A. Edwards - J. McDaniels,,,,,,3.2,22,95.5,84.7,97.2
6,F. VanVleet - D. Brooks - J. Green - A. Sengun - A. Thompson,,,,,,2.6,105,94.3,94.3,59.7
7,T. Harris - T. Hardaway Jr. - C. Cunningham - J. Duren - A. Thompson,,,,,,2.5,83,93.2,46.0,100.0
8,J. Allen - M. Strus - D. Hunter - T. Jerome - S. Merrill,,,,,,2.4,11,92.0,87.5,83.0
9,S. Curry - J. Butler III - D. Green - B. Hield - B. Podziemski,,,,,,2.3,58,90.9,93.2,53.4
10,M. Conley - J. Randle - N. Alexander-Walker - N. Reid - J. McDaniels,,,,,,2.0,10,89.8,90.3,59.7
